# Week 4 Assignment: Linear & Logistic Regression — Abalone Age and Cancer Diagnosis

In the textbook (Chapter 4) and the in-class exercise, you trained linear models on the small **Wine** dataset: the Normal Equation, Gradient Descent, Polynomial features, learning curves, regularization (Ridge/Lasso), and Logistic/Softmax classification.

In this assignment you'll apply those **same concepts** to two larger, messier real datasets and dig deeper into *how* linear models behave:

> **Regression:** Predict the number of shell rings (a proxy for **age**) of an abalone from physical measurements (`abalone` from OpenML, ~4,177 rows). Age in years ≈ rings + 1.5, so counting rings under a microscope is the slow lab method this model would replace.
>
> **Classification:** Diagnose whether a breast tumor is malignant or benign from 30 cell-nucleus measurements (`load_breast_cancer`, 569 rows).

This is harder than the in-class exercise because you will:
- Work with **thousands of rows** and a **mix of numeric and categorical** features (the abalone `Sex` column) that needs a `ColumnTransformer`
- Build a **regularization path** and pick `alpha` by cross-validation, not by hand
- Read **learning curves** to reason about bias/variance on real data
- Move beyond accuracy to **precision, recall, ROC-AUC**, and a **decision-threshold** analysis
- Interpret model **coefficients** and justify choices in writing

Adapt the techniques from the chapter — you are not expected to invent new methods.

**Data sources:** [Abalone](https://www.openml.org/search?type=data&status=active&id=183) (Nash et al., 1994; UCI via OpenML) and [Breast Cancer Wisconsin (Diagnostic)](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html) (UCI). Both download through scikit-learn.

---

### API References (scikit-learn)

| Task | Documentation |
|------|---------------|
| `fetch_openml` | [datasets.fetch_openml](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html) |
| `load_breast_cancer` | [datasets.load_breast_cancer](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html) |
| `train_test_split` | [model_selection.train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) |
| `StandardScaler` | [preprocessing.StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) |
| `OneHotEncoder` | [preprocessing.OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) |
| `ColumnTransformer` | [compose.ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) |
| `PolynomialFeatures` | [preprocessing.PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) |
| `make_pipeline` / `Pipeline` | [pipeline.make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html) |
| `LinearRegression` | [linear_model.LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) |
| `SGDRegressor` | [linear_model.SGDRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDRegressor.html) |
| `Ridge` / `RidgeCV` | [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html), [RidgeCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html) |
| `Lasso` / `LassoCV` | [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html), [LassoCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html) |
| `ElasticNet` | [linear_model.ElasticNet](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html) |
| `LogisticRegression` | [linear_model.LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) |
| `learning_curve` | [model_selection.learning_curve](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.learning_curve.html) |
| `cross_val_score` | [model_selection.cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) |
| `GridSearchCV` | [model_selection.GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) |
| Metrics (regression) | [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html), [r2_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html) |
| Metrics (classification) | [accuracy](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html), [precision/recall/f1](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html), [roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html), [roc_curve](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html) |

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/pjmcswee/IST707-Notebooks/blob/main/week4/week4_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## Setup

Run this cell to import libraries and configure plotting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml, load_breast_cancer
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     learning_curve, GridSearchCV)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.linear_model import (LinearRegression, SGDRegressor, Ridge, RidgeCV,
                                  Lasso, LassoCV, ElasticNet, LogisticRegression)
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score,
                             classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, roc_curve,
                             precision_recall_curve)

plt.rc('font', size=12)
plt.rc('axes', labelsize=13, titlesize=13)
plt.rc('legend', fontsize=11)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

RANDOM_STATE = 42

---

# PART A — Regression on Abalone Age

## A1: Load and Explore the Data (Provided)

Each row is one abalone (a marine snail). The physical measurements are cheap to take; the target `Rings` is expensive (you slice, stain, and count growth rings under a microscope). We predict `Rings`. The `Sex` column is categorical (`M`, `F`, `I` for infant). This cell is provided — just run it.

In [ ]:
abalone = fetch_openml(name='abalone', version=1, as_frame=True, parser='auto')
aba = abalone.frame.copy()

# The target ships as a 'category' dtype; cast it to a plain integer count of rings.
aba['Rings'] = aba['Class_number_of_rings'].astype(int)
aba = aba.drop(columns=['Class_number_of_rings'])

num_features = aba.select_dtypes('number').drop(columns=['Rings']).columns.tolist()
cat_features = aba.select_dtypes(['category', 'object']).columns.tolist()

print('Shape:', aba.shape)
print('Numeric features:', num_features)
print('Categorical features:', cat_features)
print('Target: Rings (age in years is approximately Rings + 1.5)')
aba.head()

In [ ]:
aba.describe().round(3)

## A2: Exploratory Data Analysis (10 points)

### A2a: Correlation with the target

**TODO:** Compute the correlation of every *numeric* feature with `Rings` and print them sorted from most positive to most negative.

**Hint:** `aba.corr(numeric_only=True)['Rings'].sort_values(ascending=False)`

In [ ]:
# TODO: Print numeric feature correlations with Rings, sorted.
# Your code here:




### A2b: Visualize the strongest relationship

**TODO:** Make a scatter plot of the single numeric feature most correlated with the target against `Rings`. Comment briefly on what you see (in the answer cell below).

**Hint:** The relationship is positive but *fans out* — older abalones vary a lot in size — which limits how well any straight line can fit.

In [ ]:
# TODO: Scatter plot of the most-correlated numeric feature vs Rings.
# Your code here:




**Question (A2c):** This is real, messy data: a couple of rows have `Height == 0`, which is physically impossible for a living abalone. Run `(aba['Height'] == 0).sum()` to count them. What are two reasonable ways to handle such invalid rows before training, and what is the risk of simply leaving them in?

*Your answer:*



## A3: Train/Test Split and Preprocessing (10 points)

Because `Sex` is categorical, a bare `StandardScaler` won't work — you need a `ColumnTransformer` that scales the numeric columns and one-hot encodes `Sex`.

**TODO:**
1. Separate features `X` (all columns except `Rings`) and target `y = aba['Rings']`.
2. Split into 80% train / 20% test with `random_state=RANDOM_STATE`.
3. Build a `ColumnTransformer` named `preprocess` with a `StandardScaler` on `num_features` and a `OneHotEncoder(handle_unknown='ignore')` on `cat_features`. Fit it on the **training** features only, then transform both train and test into `X_train_prep` / `X_test_prep`.

**Hint:**
```python
preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
])
X_train_prep = preprocess.fit_transform(X_train)
X_test_prep = preprocess.transform(X_test)
```

In [ ]:
# TODO: Build X, y; split; build and fit the ColumnTransformer (fit on train only).
# Name your variables X_train, X_test, y_train, y_test,
# preprocess, X_train_prep, X_test_prep.
# Your code here:




**Question (A3a):** Why must the scaler be fit on the training set only and then *applied* to the test set, rather than fit on the full dataset before splitting? Name the specific problem that fitting on all the data would cause.

*Your answer:*



## A4: Closed-Form vs Gradient Descent (15 points)

### A4a: LinearRegression and SGDRegressor

**TODO:** On the **preprocessed** data (`X_train_prep` / `X_test_prep`), fit both a `LinearRegression` and an `SGDRegressor`. For each, print the **test RMSE** and **test R²**. Use `SGDRegressor(max_iter=1000, tol=1e-3, penalty=None, random_state=RANDOM_STATE)`.

They should land close to each other — Gradient Descent is *approximating* the same solution the Normal Equation finds exactly.

In [ ]:
# TODO: Fit LinearRegression and SGDRegressor on preprocessed data.
#       Print test RMSE and R^2 for each.
# Your code here:




### A4b: Timing comparison

**TODO:** Use `%timeit` (or `time.perf_counter`) to compare how long it takes to `.fit()` `LinearRegression` vs `SGDRegressor` on this ~4k-row dataset. Report both times.

**Hint:** `%timeit -n 3 -r 3 LinearRegression().fit(X_train_prep, y_train)`

In [ ]:
# TODO: Time LinearRegression.fit vs SGDRegressor.fit (on X_train_prep).
# Your code here:




**Question (A4c):** The Normal Equation has computational complexity roughly $O(n \cdot m^2)$ in the number of features $m$ and is exact, while Gradient Descent is iterative. On *this* dataset (about 10 features after encoding, several thousand rows) which approach do you expect to win on speed, and how would that change if you had 100,000 features instead of 10?

*Your answer:*



## A5: Polynomial Features and Learning Curves (15 points)

### A5a: Does adding polynomial features help?

**TODO:** Build three pipelines of `preprocess` + `PolynomialFeatures(degree=d)` + `LinearRegression` for `d = 1, 2, 3` and report the **5-fold cross-validation RMSE** (mean and std) for each, using the raw `X_train` (let the pipeline preprocess). Use `scoring='neg_root_mean_squared_error'`.

**Hint:** Reuse your fitted-from-scratch transformer by cloning it, or just build a fresh `ColumnTransformer` inside the pipeline:
```python
from sklearn.base import clone
model = make_pipeline(clone(preprocess),
                      PolynomialFeatures(degree=d, include_bias=False),
                      LinearRegression())
```

In [ ]:
# TODO: For degree in [1, 2, 3], build pipeline and report CV RMSE mean/std.
# Your code here:




### A5b: Learning curves

**TODO:** Plot learning curves (train RMSE and validation RMSE vs training-set size) for a **degree-1** and a **degree-3** polynomial pipeline (each starting with `clone(preprocess)`) on `X_train`, `y_train`. Put both on the same figure (or side-by-side subplots) so the bias/variance contrast is visible.

**Hint (from the in-class exercise):**
```python
train_sizes, train_scores, valid_scores = learning_curve(
    model, X_train, y_train, train_sizes=np.linspace(0.05, 1.0, 20),
    cv=5, scoring='neg_root_mean_squared_error')
train_errors = -train_scores.mean(axis=1)
valid_errors = -valid_scores.mean(axis=1)
```

In [ ]:
# TODO: Plot learning curves for degree-1 and degree-3 pipelines.
# Your code here:




**Question (A5c):** From your learning curves, is the degree-1 model more limited by **bias** or **variance**? What about degree-3? Point to the specific behavior of the curves (the gap between them, and where each plateaus) that supports your answer.

*Your answer:*



## A6: Regularization Path and Tuning (15 points)

### A6a: Ridge coefficient path

**TODO:** For a range of `alpha` values (e.g. `np.logspace(-3, 4, 30)`), fit `Ridge` on the preprocessed training data (`X_train_prep`) and record each fitted coefficient vector. Plot each coefficient's value as a function of `alpha` on a log-x axis (one line per feature). This reproduces the textbook's "coefficients shrink toward zero as regularization increases" figure.

**Hint:** Collect coefficients into a list, `np.array` it (shape `[n_alphas, n_features]`), and plot column by column. Get the feature names after preprocessing with `preprocess.get_feature_names_out()`.

In [ ]:
# TODO: Fit Ridge across a log-spaced alpha grid; plot the coefficient paths.
# Your code here:




### A6b: Pick alpha by cross-validation

**TODO:** Use `RidgeCV` and `LassoCV` (each with an `alphas` grid and `cv=5`) on the preprocessed training data (`X_train_prep`) to select the best `alpha` automatically. For each, print:
- the selected `alpha_`
- the test RMSE
- for Lasso: how many coefficients were driven to exactly 0, and which features survived.

**Hint:** `RidgeCV(alphas=np.logspace(-3, 4, 30), cv=5)`; `LassoCV(alphas=np.logspace(-3, 1, 30), cv=5, max_iter=5000)`.

In [ ]:
# TODO: RidgeCV and LassoCV; print selected alpha, test RMSE, and Lasso sparsity.
# Your code here:




**Question (A6c):** Look at the Ridge coefficient path. As `alpha` grows very large, what value do all coefficients approach, and what does the model's prediction become in that limit? Connect this to the idea that regularization trades variance for bias.

*Your answer:*



**Question (A6d):** Did Lasso set any coefficients to exactly 0? If so, are the dropped features the ones you'd expect from the correlation analysis in A2? If Lasso kept everything, explain why that can happen on this dataset.

*Your answer:*



---

# PART B — Logistic Regression on Breast Cancer Diagnosis

## B1: Load the Data (Provided)

Each row describes a tumor with 30 numeric features. The target is `0 = malignant`, `1 = benign` (this is the scikit-learn convention). This cell is provided — just run it.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
cdf = cancer.frame

print('Shape:', cdf.shape)
print('Target names (index = label):', list(cancer.target_names))  # 0=malignant, 1=benign
print('Class balance:')
print(cdf['target'].value_counts())
cdf.head()

## B2: Prepare, Train, and Baseline (15 points)

### B2a: Split and scale

**TODO:**
1. Build `Xc` (the 30 features) and `yc` (`target`).
2. Split 75% train / 25% test with `random_state=RANDOM_STATE` and **stratify on `yc`**.
3. Scale with a `StandardScaler` fit on the training set only.

**Hint:** `train_test_split(..., stratify=yc, random_state=RANDOM_STATE)`.

In [ ]:
# TODO: Build Xc, yc; stratified split; scale (fit on train only).
# Name them Xc_train_scaled, Xc_test_scaled, yc_train, yc_test.
# Your code here:




**Question (B2b):** Why did we pass `stratify=yc` to the split? What could go wrong on a class-imbalanced dataset if we didn't stratify?

*Your answer:*



### B2c: Train a logistic regression

**TODO:** Train a `LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)` on the scaled training data. Print the **test accuracy** and the full **classification report** (precision, recall, f1 per class).

**Hint:** `print(classification_report(yc_test, y_pred, target_names=cancer.target_names))`.

In [ ]:
# TODO: Train LogisticRegression; print accuracy and classification_report.
# Your code here:




## B3: Regularization Strength `C` (10 points)

### B3a: Tune C

`LogisticRegression` has an inverse-regularization parameter `C` (small `C` = strong regularization). 

**TODO:** Use `GridSearchCV` (5-fold) over `C` in `np.logspace(-3, 3, 13)` with `penalty='l2'`, scoring on `'roc_auc'`. Report the best `C` and the best cross-validated ROC-AUC.

**Hint:** Search on the **scaled training** data; put the estimator directly (no pipeline needed since the data is already scaled), with `param_grid={'C': np.logspace(-3, 3, 13)}`.

In [ ]:
# TODO: GridSearchCV over C; report best C and best CV ROC-AUC.
# Your code here:




**Question (B3b):** As you decrease `C` (stronger regularization), what happens to the magnitude of the logistic-regression coefficients, and what is the risk of setting `C` too small on this problem?

*Your answer:*



## B4: Beyond Accuracy — ROC and Decision Threshold (15 points)

### B4a: ROC curve and AUC

**TODO:** Using your best model from B3 (refit on the full scaled training set), get predicted **probabilities** for the positive class on the test set, then plot the **ROC curve** and print the **ROC-AUC**.

**Hint:**
```python
y_proba = best_model.predict_proba(Xc_test_scaled)[:, 1]
fpr, tpr, thr = roc_curve(yc_test, y_proba)
auc = roc_auc_score(yc_test, y_proba)
```

In [ ]:
# TODO: Plot ROC curve and print ROC-AUC.
# Your code here:




### B4b: Moving the decision threshold

In a medical setting, missing a **malignant** tumor (a false negative for cancer) is far worse than a false alarm. Recall that here the label `0 = malignant`. The default classifier threshold of 0.5 on `P(benign)` may not be the right operating point.

**TODO:**
1. Define the *malignant* probability as `p_malignant = 1 - y_proba` (since `y_proba` is `P(benign)`).
2. Classify a tumor as malignant when `p_malignant >= t`. For thresholds `t` in `[0.1, 0.3, 0.5, 0.7, 0.9]`, print the number of **missed malignancies** (malignant tumors predicted benign) and the number of **false alarms** (benign tumors flagged malignant).
3. Briefly recommend a threshold for this use case in the answer cell.

**Hint:** Build a boolean mask `pred_malignant = p_malignant >= t`, and compare against the true malignant mask `(yc_test == 0)`.

In [ ]:
# TODO: Sweep the malignant-detection threshold; print missed malignancies
#       and false alarms at each threshold.
# Your code here:




**Question (B4c):** Based on your threshold sweep, which threshold would you deploy for a cancer *screening* tool, and what is the trade-off you are accepting by choosing it? Explain in terms of false negatives vs false positives.

*Your answer:*



### B4d: Which features drive the prediction?

**TODO:** Because the features were standardized, the logistic-regression coefficients are directly comparable in magnitude. Print the **5 features with the largest positive** coefficients and the **5 with the largest negative** coefficients, with their names.

**Hint:** Pair `best_model.coef_[0]` with `cancer.feature_names` in a DataFrame and sort.

In [ ]:
# TODO: Rank features by coefficient magnitude and print top positive/negative.
# Your code here:




**Question (B4e):** Recall that the positive class is `1 = benign`. A feature with a large **negative** coefficient pushes the prediction toward which diagnosis? Name one such top feature and give a one-sentence intuition for why that direction makes clinical sense.

*Your answer:*



---

## Part C: Reflection (Ungraded but Required)

Answer each question in 3-5 sentences.

**Q1:** Compare this assignment's datasets to the in-class Wine dataset. Name two concrete ways that having ~4,000 rows with a categorical feature (Abalone) instead of 178 all-numeric rows (Wine) changed which techniques were worth using or how the learning curves looked.

*Your answer:*



**Q2:** Ridge and Lasso both fought overfitting in Part A. Based on your results, when would you reach for Lasso over Ridge on a problem like the Abalone data, and when would Ridge be the safer default?

*Your answer:*



**Q3:** In Part B you moved past accuracy to ROC-AUC and a custom threshold. Explain why accuracy alone can be a misleading metric for a diagnostic classifier, and what ROC-AUC captures that accuracy does not.

*Your answer:*



**Q4:** Both linear and logistic regression are *linear* models, yet one predicts a continuous value and the other a probability. Explain the single key transformation that turns the linear combination $\theta^\top x$ into a valid probability, and why we can't just use the raw linear output as a probability.

*Your answer:*



---

## Grading Rubric

| Section | Points |
|---------|--------|
| A2: EDA (correlation, plot, data-quality question) | 10 |
| A3: Split & preprocessing (with leakage question) | 10 |
| A4: Closed-form vs Gradient Descent (+ timing) | 15 |
| A5: Polynomial features & learning curves | 15 |
| A6: Regularization path & CV tuning | 15 |
| B2: Prepare, train, baseline classifier | 15 |
| B3: Tuning regularization strength C | 10 |
| B4: ROC, threshold analysis, coefficients | 15 |
| **Total** | **100** |

Part C (Reflection) is ungraded but **required for full credit** — incomplete reflections will result in a 5-point deduction.